# Notebook 3 – Missing Value Handling

## 1. What are missing values and why do they occur?
Missing values represent information that is absent, unrecorded, unavailable, or not applicable. Causes include skipped forms, system errors, data merges, and measurement failure.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "Age": [25, np.nan, 40, 35, np.nan, 29],
    "MonthlyCharges": [50, 70, np.nan, 90, 65, 55],
    "Contract": ["Monthly", "Yearly", np.nan, "Monthly", "Yearly", "Monthly"],
    "Region": ["A", "A", "B", "B", "B", "A"]
})
display(df)
print("Missing counts:\n", df.isna().sum())
print("Missing percentage:\n", (df.isna().mean() * 100).round(2))

,Age,MonthlyCharges,Contract,Region
0,25.0,50.0,Monthly,A
1,NaN,70.0,Yearly,A
2,40.0,NaN,NaN,B
3,35.0,90.0,Monthly,B
4,NaN,65.0,Yearly,B
5,29.0,55.0,Monthly,A


Missing counts:
 Age               2
MonthlyCharges    1
Contract          1
Region            0
dtype: int64
Missing percentage:
 Age               33.33
MonthlyCharges    16.67
Contract          16.67
Region             0.00
dtype: float64


**Output explanation:** `isna()` counts recognized missing values. The percentage helps prioritize columns but does not by itself tell you whether deletion or imputation is best.

## 2. MCAR, MAR, and MNAR
- **MCAR:** Missingness is unrelated to observed or unobserved data.
- **MAR:** Missingness depends on other observed variables.
- **MNAR:** Missingness depends on the missing value itself or unobserved information.

**Example:** A random sensor outage might be MCAR; missing income associated with observed employment type may be MAR; people with very high income choosing not to report it may be MNAR. These assumptions require context and cannot usually be proven from null counts alone.

## 3. Dropping rows or columns
**Use row deletion** when only a small, plausibly random portion is missing and enough data remains. **Avoid it** when it removes many records or creates bias. **Drop a column** only when it is not useful or missingness is too extensive to justify recovery; assess business importance first.

In [2]:
row_dropped = df.dropna(subset=["Age"])
col_dropped = df.drop(columns=["Region"])  # illustrative only; not recommended without analysis
print("Original shape:", df.shape)
print("Rows after dropping missing Age:", row_dropped.shape)
print("Columns after illustrative column drop:", col_dropped.shape)

Original shape: (6, 4)
Rows after dropping missing Age: (4, 4)
Columns after illustrative column drop: (6, 3)


**Output explanation:** Dropping rows reduces observations; dropping columns reduces available features. These are demonstrations, not recommendations for the real dataset.

## 4. Mean, median, mode, and constant imputation
- **Mean:** useful for roughly symmetric numerical data; sensitive to outliers.
- **Median:** robust to outliers and useful for skewed numerical data; can reduce natural variation.
- **Mode:** useful for categorical data; may overrepresent the most common category.
- **Constant:** use a meaningful value such as `Unknown` for categories when appropriate; arbitrary numeric constants can distort relationships.

Do not fill every missing value with the mean.

In [3]:
from sklearn.impute import SimpleImputer

num_mean = SimpleImputer(strategy="mean")
num_median = SimpleImputer(strategy="median")
cat_mode = SimpleImputer(strategy="most_frequent")
cat_constant = SimpleImputer(strategy="constant", fill_value="Unknown")

print("Mean Age:", num_mean.fit_transform(df[["Age"]]).ravel())
print("Median Age:", num_median.fit_transform(df[["Age"]]).ravel())
print("Mode Contract:", cat_mode.fit_transform(df[["Contract"]]).ravel())
print("Constant Contract:", cat_constant.fit_transform(df[["Contract"]]).ravel())

Mean Age: [25.   32.25 40.   35.   32.25 29.  ]
Median Age: [25. 32. 40. 35. 32. 29.]
Mode Contract: ['Monthly' 'Yearly' 'Monthly' 'Monthly' 'Yearly' 'Monthly']
Constant Contract: ['Monthly' 'Yearly' 'Unknown' 'Monthly' 'Yearly' 'Monthly']


**Output explanation:** Each imputer learns a replacement from the supplied data. In real ML workflows, fit imputers on training data only, then transform validation/test data.

## 5. Forward fill, backward fill, and interpolation
These methods are mainly appropriate when row order has meaning, especially time series. Forward fill uses the previous value; backward fill uses the next value; interpolation estimates between known values. Avoid them for unordered customer rows unless a valid ordering and business reason exist.

In [4]:
series = pd.Series([10.0, np.nan, np.nan, 40.0, np.nan])
print("Forward fill:", series.ffill().tolist())
print("Backward fill:", series.bfill().tolist())
print("Linear interpolation:", series.interpolate().tolist())

Forward fill: [10.0, 10.0, 10.0, 40.0, 40.0]
Backward fill: [10.0, 40.0, 40.0, 40.0, nan]
Linear interpolation: [10.0, 20.0, 30.0, 40.0, 40.0]


**Output explanation:** The methods produce different assumptions about missing values. Backward fill can use future information and may be inappropriate for real-time prediction.

## 6. Group-based imputation
Use group-based values when groups have meaningful differences, such as median charges by contract type. It may be unstable for small groups and must be designed to avoid leakage.

In [5]:
group_demo = pd.DataFrame({
    "Contract": ["Monthly", "Monthly", "Yearly", "Yearly", "Monthly"],
    "Charge": [50.0, np.nan, 80.0, np.nan, 60.0]
})
group_demo["Charge_filled"] = group_demo["Charge"].fillna(
    group_demo.groupby("Contract")["Charge"].transform("median")
)
display(group_demo)

,Contract,Charge,Charge_filled
0,Monthly,50.0,50.0
1,Monthly,NaN,55.0
2,Yearly,80.0,80.0
3,Yearly,NaN,80.0
4,Monthly,60.0,60.0


**Output explanation:** Missing charges are filled using the median for the same contract group. If a group has no observed values, a fallback strategy is needed.

## 7. KNN and Iterative imputation
**KNN imputation** estimates missing values from similar rows; it can be computationally expensive and sensitive to feature scales. **Iterative imputation** models each incomplete feature using other features; it can capture relationships but takes longer and relies on modeling assumptions.

In [6]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import KNNImputer, IterativeImputer

numeric_demo = pd.DataFrame({
    "Age": [25, np.nan, 40, 35],
    "Charge": [50, 70, np.nan, 90]
})

knn = KNNImputer(n_neighbors=2)
iterative = IterativeImputer(random_state=42, max_iter=10)

print("KNN:")
display(pd.DataFrame(
    knn.fit_transform(numeric_demo),columns=numeric_demo.columns ))
prin t("Iterative:")
display(pd.DataFrame(iterative.fit_transform(numeric_demo), columns=numeric_demo.columns))

ImportError: IterativeImputer is experimental and the API might change without any deprecation cycle. To use it, you need to explicitly import enable_iterative_imputer:
from sklearn.experimental import enable_iterative_imputer

**Output explanation:** Both methods estimate missing numerical values using patterns among columns. Scale features thoughtfully for KNN, evaluate imputation quality, and fit only on training data in a predictive workflow.

## 8. Choosing a technique: advantages and limitations
| Technique | Use when | Avoid/caution | Advantage | Limitation |
|---|---|---|---|---|
| Drop rows | Few rows missing at random | Missingness is systematic | Simple | Loses data |
| Drop columns | Feature is unneeded or unusable | Feature is business-critical | Simplifies data | Loses information |
| Mean | Numeric, roughly symmetric | Strong outliers/skew | Easy | Outlier-sensitive |
| Median | Skewed numeric data | Relationships need preservation | Robust | Reduces variation |
| Mode | Categorical data | Many equally common classes | Simple | Dominant-category bias |
| Constant | Meaningful unknown category | Arbitrary numeric constant | Explicit missingness | Can create artificial group |
| Forward/back fill | Ordered data | Unordered rows or leakage risk | Preserves sequence | Carries old/future values |
| Interpolation | Continuous ordered series | Nominal categories | Smooth estimate | Assumes trend |
| Group-based | Meaningful groups | Tiny/empty groups | Uses context | Group-specific bias |
| KNN | Similar rows are informative | Large/unscaled data | Uses local similarity | Cost and scale sensitivity |
| Iterative | Features predict each other | Weak assumptions/data | Models relationships | More complex |

### Review questions
1. Explain MCAR, MAR, and MNAR.
2. How would you handle 40% missing values?
3. Mean vs median imputation?
4. When is forward fill appropriate?
5. Why should an imputer be fitted on training data only?